# 03 - Tabla gold: pasajeros por línea, día y hora

Arma `demanda_gold`, la tabla con la que se entrenan los modelos. **Cada fila es una línea, un día y una hora.** Por ejemplo: línea 103, martes 12 de marzo de 2024, de 8:00 a 8:59.

| Paso | Qué hace |
|---|---|
| 1. Sumar | Suma los pasajeros de silver por línea, día y hora |
| 2. Elegir líneas | Se queda con las líneas que funcionaron al menos `MIN_DIAS_LINEA` días en **cada** año de train y val (2024 y 2025). 2026 no se mira |
| 3. Grilla | Arma todas las combinaciones línea x día x hora. Si en una hora no subió nadie, pone **0** |
| 4. Calendario | Agrega día de la semana, mes, feriados, Carnaval, Turismo, elecciones, días sin ómnibus, paros de transporte, fiestas de fin de año (24 al 31 de diciembre), si hay clases en escuelas y liceos, y qué tipo de día fue ayer |
| 4b. Clima | Agrega la temperatura y la lluvia **que pronosticaban el día anterior** (promedio de 4 servicios del tiempo) |
| 5. Rezagos | Agrega cuántos pasajeros hubo en la misma línea y hora 1, 7, 14, 21 y 28 días antes, el promedio y la mediana de esas 4 semanas, y el total de la línea el día anterior |
| 6. Dividir | Marca cada fila como `train` (2024), `val` (2025) o `test` (2026) según la fecha |

**Regla importante.** Queremos predecir un día usando solo lo que se sabe la noche anterior. Por eso los rezagos solo miran días **anteriores**, y del clima usamos el pronóstico y no lo que pasó de verdad. Si usáramos datos del mismo día, el modelo "haría trampa": en la vida real esos datos todavía no existen.

**Paros.** La columna `es_paro` marca los paros de transporte que se notaron en los datos. Por ahora es uno solo, el de la Unott del 31/07/2025. En los paros generales del PIT-CNT los ómnibus anduvieron normal (ver `calendario.py`). La tabla de "Días raros" del final sirve para ver si quedó algún día bajo sin explicación.

**No está:** el calendario de UdelaR (cada facultad tiene el suyo).

In [0]:
%run ./00_setup

# 00 - Preparar el proyecto

Este notebook deja todo listo para que los demás funcionen. Hace cuatro cosas:

1. Le dice a Python dónde está nuestro código (la carpeta `src/bondis`), para poder escribir `from bondis import ...`.
2. Crea en Databricks el lugar donde se guardan las tablas y los archivos (`bondis_stm.datos.raw`, definido en `config.py`).
3. Crea una carpeta para cada tipo de archivo (viajes, clima, etc.).
4. Muestra cómo dividimos los datos en train, val y test.

Se puede correr todas las veces que quieras. Si algo ya existe, no lo vuelve a crear ni lo borra.

No hace falta abrirlo a mano: los otros notebooks lo corren al principio con `%run ./00_setup`.

## 1. Encontrar nuestro código

Python solo puede importar código de las carpetas que conoce. Esta celda busca la carpeta del repositorio subiendo carpeta por carpeta hasta encontrar `src/bondis`, y la agrega a la lista de carpetas que Python conoce (`sys.path`).

## 2. Crear el catálogo, el esquema y el volume

Databricks ordena todo en tres niveles, como carpetas una dentro de otra:

- **Catálogo** (`bondis_stm`): la carpeta grande del proyecto.
- **Esquema** (`datos`): una subcarpeta donde van las tablas.
- **Volume** (`raw`): una carpeta para guardar archivos sueltos, como los ZIP y CSV que bajamos.


## 3. Crear una carpeta para cada tipo de archivo

| Carpeta | Qué va adentro |
|---|---|
| `viajes_zip/` | Los ZIP de viajes de cada mes, bajados a mano del portal de la Intendencia |
| `viajes/` | Los CSV de viajes, ya sacados de los ZIP |
| `clima/` | Lluvia y temperatura de cada hora |
| `calendario/` | Por ahora vacía. El calendario está escrito en `src/bondis/calendario.py` |
| `paros/` | Días y horas con paro (todavía no lo hicimos) |

## 4. Revisar que todo esté bien

Muestra la versión de Spark, el usuario y cuántos archivos hay en cada carpeta. Sirve para ver de un vistazo si falta subir algo.

## 5. Cómo dividimos los datos

Usamos las fechas para dividir, no el azar, porque queremos predecir el futuro con el pasado. Un año entero para cada cosa:

| Conjunto | Fechas | Para qué se usa |
|---|---|---|
| `train` | todo 2024 | El modelo aprende con estos datos |
| `val` | todo 2025 | Comparamos y elegimos los parámetros de la regresión |
| `test` | 2026 (lo que haya) | Probamos el modelo elegido una sola vez, al final |

La fecha de fin **no** está incluida. Por eso siempre se filtra con `fecha >= inicio` y `fecha < fin`.

`train_final` es train + val juntos. Cuando ya elegimos el modelo, lo volvemos a entrenar con 2024 y 2025 antes de probarlo en 2026.

... (14 outputs left out)

## 2. Crear el catálogo, el esquema y el volume

Databricks ordena todo en tres niveles, como carpetas una dentro de otra:

- **Catálogo** (`bondis_stm`): la carpeta grande del proyecto.
- **Esquema** (`datos`): una subcarpeta donde van las tablas.
- **Volume** (`raw`): una carpeta para guardar archivos sueltos, como los ZIP y CSV que bajamos.


## 3. Crear una carpeta para cada tipo de archivo

| Carpeta | Qué va adentro |
|---|---|
| `viajes_zip/` | Los ZIP de viajes de cada mes, bajados a mano del portal de la Intendencia |
| `viajes/` | Los CSV de viajes, ya sacados de los ZIP |
| `clima/` | Lluvia y temperatura de cada hora |

El calendario (feriados, clases y paros) no tiene carpeta, porque está escrito a mano en `src/bondis/calendario.py`.

## 4. Revisar que todo esté bien

Muestra la versión de Spark y cuántos archivos hay en cada carpeta. Sirve para ver de un vistazo si falta subir algo.

## 5. Cómo dividimos los datos

Usamos las fechas para dividir, no el azar, porque queremos predecir el futuro con el pasado. Un año entero para cada cosa:

| Conjunto | Fechas | Para qué se usa |
|---|---|---|
| `train` | todo 2024 | El modelo aprende con estos datos |
| `val` | todo 2025 | Comparamos y elegimos los parámetros de la regresión |
| `test` | 2026 (lo que haya) | Probamos el modelo elegido una sola vez, al final |

La fecha de fin **no** está incluida. Por eso siempre se filtra con `fecha >= inicio` y `fecha < fin`.

`train_final` es train + val juntos. Cuando ya elegimos el modelo, lo volvemos a entrenar con 2024 y 2025 antes de probarlo en 2026.

In [0]:
import importlib
from datetime import date, timedelta

from pyspark.sql import functions as F

from bondis import calendario, clima, gold

importlib.reload(calendario)
importlib.reload(clima)
importlib.reload(gold)

spark.conf.set("spark.sql.session.timeZone", config.TIMEZONE)

## 1. Sumar los pasajeros de silver

Silver tiene unos 818 millones de filas (una por cada viaje). Acá se suman por línea, día y hora. Es el paso más lento, por eso el resultado se guarda en una tabla aparte (`demanda_agregada`) y así no hay que repetirlo.

Solo se usan las fechas del proyecto (de 2024 a 2026, ver `config.FECHA_FIN`). Si 2026 no está completo, gold llega hasta el último día que haya en silver.

In [0]:
TABLE_AGREGADA = f"{config.CATALOG}.{config.SCHEMA}.demanda_agregada"

silver_periodo = spark.table(config.TABLE_SILVER).where(
    (F.col("fecha") >= F.to_date(F.lit(config.FECHA_INICIO)))
    & (F.col("fecha") < F.to_date(F.lit(config.FECHA_FIN)))
)
(
    gold.agregar(silver_periodo)
    .write.format("delta").mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(TABLE_AGREGADA)
)
agregado = spark.table(TABLE_AGREGADA)

rango = agregado.agg(F.min("fecha").alias("ini"), F.max("fecha").alias("fin"),
                     F.sum("pasajeros").alias("pasajeros"), F.count("*").alias("filas")).first()
print(f"Filas agregadas:   {rango['filas']:,}")
print(f"Pasajeros totales: {rango['pasajeros']:,}")
print(f"Fechas:            {rango['ini']} a {rango['fin']}")

Filas agregadas:   2,577,441
Pasajeros totales: 848,033,192
Fechas:            2024-01-01 a 2026-10-31


## 2. Elegir las líneas

Se dejan afuera las líneas que funcionaron menos de 100 días en 2024 o en 2025 (por ejemplo D1 y L8, que arrancaron en 2025, o BT1 y BT2, que arrancaron a fines de 2024). Esas líneas no tienen suficientes datos para aprender. La tabla de abajo muestra qué líneas quedaron afuera y cuántos pasajeros representan.

**Para elegir las líneas se mira 2024 y 2025, pero no 2026 (test).** Se usa 2025 porque una línea que casi no anduvo en val no se puede validar. Es una decisión que toca val, pero no test. Si exigiéramos que una línea funcione también en 2026, estaríamos usando datos del test para decidir qué entra al modelo, y eso es hacer trampa: el día que se usa el modelo no sabemos qué líneas van a seguir funcionando. Por eso, una línea que dejó de andar en 2026 queda en gold con 0 pasajeros esos días, igual que pasaría en la realidad. Y por lo mismo la 468, que arrancó en 2026, queda afuera aunque lleve muchos pasajeros.

In [0]:
resumen = gold.resumen_lineas(agregado)
# Solo los años de train y val. El año de test (2026) no se usa para elegir líneas.
anio_test = date.fromisoformat(config.FECHA_INICIO_TEST).year
anios = sorted(r["anio"] for r in resumen.select("anio").distinct().collect() if r["anio"] < anio_test)
print("Años que se miran para elegir líneas:", anios)
lineas = gold.lineas_incluidas(resumen, anios, config.MIN_DIAS_LINEA)

lineas.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(config.TABLE_LINEAS)
lineas = spark.table(config.TABLE_LINEAS)

stats = lineas.groupBy("incluida").agg(F.count("*").alias("lineas"),
                                       F.sum("pasajeros_total").alias("pasajeros")).collect()
total = sum(r["pasajeros"] for r in stats)
for r in sorted(stats, key=lambda r: not r["incluida"]):
    etiqueta = "Incluidas" if r["incluida"] else "Excluidas"
    print(f"{etiqueta}: {r['lineas']:>4} líneas, {r['pasajeros']:>13,} pasajeros ({100 * r['pasajeros'] / total:.2f} %)")

print("\nLíneas excluidas:")
display(lineas.where(~F.col("incluida")).orderBy(F.desc("pasajeros_total")))

Años que se miran para elegir líneas: [2024, 2025]
Incluidas:  136 líneas,   843,853,308 pasajeros (99.51 %)
Excluidas:   11 líneas,     4,179,884 pasajeros (0.49 %)

Líneas excluidas:


linea,dias_2024,dias_2025,pasajeros_total,incluida,motivo
468,null,null,3121619,false,menos de 100 días de servicio en algún año
D1,null,233,494238,false,menos de 100 días de servicio en algún año
L8,null,261,203935,false,menos de 100 días de servicio en algún año
BT1,53,363,173603,false,menos de 100 días de servicio en algún año
L77,null,null,115636,false,menos de 100 días de servicio en algún año
BT2,46,363,47381,false,menos de 100 días de servicio en algún año
PA,2,2,12144,false,menos de 100 días de servicio en algún año
PRUEX_L8,null,16,4566,false,menos de 100 días de servicio en algún año
PB,2,2,4315,false,menos de 100 días de servicio en algún año
M-A,2,1,2205,false,menos de 100 días de servicio en algún año


## 3. Grilla completa y calendario

¿Por qué una grilla? Silver solo tiene filas cuando alguien subió al ómnibus. Si a las 3 de la mañana nadie subió a la línea 103, esa hora no aparece. Pero para el modelo "0 pasajeros" es un dato importante. La grilla arma **todas** las horas de todos los días para cada línea y rellena con 0 las que no tienen viajes.

Después se le pegan las columnas del calendario (feriados, clases, etc.) y el tipo de línea.

In [0]:
fecha_inicio = date.fromisoformat(config.FECHA_INICIO)
fecha_fin = min(rango["fin"], date.fromisoformat(config.FECHA_FIN) - timedelta(days=1))

cal = calendario.tabla_calendario(spark, fecha_inicio, fecha_fin)
cal.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(config.TABLE_CALENDARIO)

base = gold.completar(
    gold.grilla(spark, lineas.where("incluida"), fecha_inicio, fecha_fin),
    agregado,
)
base = gold.agregar_variables_fecha(base, spark.table(config.TABLE_CALENDARIO))
base = gold.tipo_linea(base)

### Clima

Solo las columnas **pronosticadas** (las que terminan en `_pron`) entran en gold, porque son las que sabríamos la noche anterior. Lo que llovió de verdad (`_obs`) queda en la tabla `clima` solo para ver qué tan bien acierta el pronóstico. No se usa en el modelo.

La tabla de abajo muestra desde qué día hay pronóstico, en qué porcentaje de horas hay dato, cuántos grados se equivoca en promedio y en qué porcentaje de horas pronosticó lluvia y llovió de verdad.

In [0]:
archivos_clima = clima.clasificar_archivos(config.RAW_CLIMA)
for tipo, lista in archivos_clima.items():
    print(f"{tipo}: {[a.split('/')[-1] for a in lista]}")
if archivos_clima["no_reconocido"]:
    print("ATENCIÓN: los archivos no reconocidos no se usan. Hay que abrirlos para ver qué tienen.")

clima_df, columnas_modelos = clima.tabla_clima(spark, archivos_clima["pronostico"], archivos_clima["observado"])
clima_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(config.TABLE_CLIMA)
print("Columnas de pronóstico promediadas:", columnas_modelos)

# Control: cada partición tiene que tener pronóstico del clima. Si a una le falta un pedazo,
# esas filas quedan incompletas y los modelos las dejan afuera sin avisar.
# Contamos desde el primer día con pronóstico guardado (19/01/2024), porque lo que falta
# antes de ese día ya lo sabemos y no es un error.
clima_tabla = spark.table(config.TABLE_CLIMA)
primer_pron = clima_tabla.where(F.col("modelos_disponibles") > 0).agg(F.min("fecha")).first()[0]
if primer_pron is None:
    raise ValueError("La tabla de clima no tiene ninguna hora con pronóstico. Hay que revisar los archivos de raw/clima.")

print(f"\nPronóstico por partición (contando desde el {primer_pron}, el primer día con pronóstico guardado):")
for nombre, (inicio, fin) in config.PARTICIONES.items():
    hasta = min(date.fromisoformat(fin) - timedelta(days=1), fecha_fin)
    if date.fromisoformat(inicio) > hasta:
        print(f"  {nombre:<5} sin datos de viajes todavía")
        continue
    desde = max(date.fromisoformat(inicio), primer_pron)
    horas = 0 if desde > hasta else (
        clima_tabla.where(F.col("fecha").between(desde, hasta))
        .agg(F.sum((F.col("modelos_disponibles") > 0).cast("int"))).first()[0] or 0)
    if horas == 0:
        raise ValueError(
            f"La partición {nombre} no tiene pronóstico del clima. Falta subir a raw/clima el "
            f"archivo de pronóstico que cubra {inicio[:4]} (los enlaces están en src/bondis/clima.py)."
        )
    total = ((hasta - desde).days + 1) * 24
    print(f"  {nombre:<5} {desde} a {hasta}: {100 * horas / total:5.1f} % de las horas con pronóstico")
    if horas < 0.95 * total:
        print(f"        ATENCIÓN: a {nombre} le falta pronóstico en más del 5 % de las horas. Esas filas "
              "no las usa ningún modelo. Hay que revisar que el archivo de pronóstico llegue hasta el último día con viajes.")

COLUMNAS_CLIMA = ["temperatura_pron", "lluvia_pron_mm", "llueve_pron", "lluvia_pron_dia_mm"]
base = base.join(
    spark.table(config.TABLE_CLIMA).select("fecha", "hora", *COLUMNAS_CLIMA),
    ["fecha", "hora"], "left",
)

display(
    spark.table(config.TABLE_CLIMA)
    .where(F.col("fecha").between(fecha_inicio, fecha_fin))
    .agg(F.min(F.when(F.col("modelos_disponibles") > 0, F.col("fecha"))).alias("primer_pronostico"),
         F.round(F.avg((F.col("modelos_disponibles") > 0).cast("int")) * 100, 1).alias("pct_horas_con_pronostico"),
         F.round(F.avg(F.abs(F.col("temperatura_pron") - F.col("temperatura_obs"))), 2).alias("error_medio_temp_C"),
         F.round(F.avg(F.col("llueve_pron").cast("int")) * 100, 1).alias("pct_horas_lluvia_pron"),
         F.round(F.avg((F.col("lluvia_obs_mm") >= clima.UMBRAL_LLUVIA_MM).cast("int")) * 100, 1).alias("pct_horas_lluvia_obs"))
)

pronostico: ['open-meteo-34.85S56.13W51m (6).csv']
observado: ['open-meteo-34.90S56.19W51m (1).csv']
no_reconocido: []
Columnas de pronóstico promediadas: ['temperature_2m_previous_day1_gfs_global', 'temperature_2m_previous_day1_ecmwf_ifs025', 'temperature_2m_previous_day1_icon_global', 'temperature_2m_previous_day1_gem_global', 'precipitation_previous_day1_gfs_global', 'precipitation_previous_day1_ecmwf_ifs025', 'precipitation_previous_day1_icon_global', 'precipitation_previous_day1_gem_global']

Pronóstico por partición:
  train 2024-01-01 a 2024-12-31:  95.0 % de las horas con pronóstico
  val   2025-01-01 a 2025-12-31: 100.0 % de las horas con pronóstico
  test  2026-01-01 a 2026-10-31: 100.0 % de las horas con pronóstico


primer_pronostico,pct_horas_con_pronostico,error_medio_temp_C,pct_horas_lluvia_pron,pct_horas_lluvia_obs
2024-01-19,98.2,1.65,12.6,12.2


## 4. Rezagos, división en train/val/test y guardar gold

**¿Qué es un rezago?** Es mirar para atrás. Para la línea 103 el martes 12 a las 8:
- `pasajeros_lag_1d` = pasajeros del lunes 11 a las 8 (1 día antes)
- `pasajeros_lag_7d` = pasajeros del martes 5 a las 8 (1 semana antes)
- `pasajeros_mediana_4sem` = la mediana de los martes 5 de marzo, 27, 20 y 13 de febrero, a las 8

Usamos rezagos porque la mejor pista de cuánta gente va a viajar es cuánta gente viajó antes en la misma situación.

**Filas completas.** Las primeras semanas de 2024 no tienen rezago de 28 días (no hay datos de diciembre de 2023), y antes del 19 de enero tampoco hay pronóstico del clima. Además sacamos los días en que una línea todavía no existía y su primer mes (columna `linea_en_marcha`). La grilla les pone 0 pasajeros, pero ese 0 no es un dato de verdad, es que la línea no andaba. Las líneas que ya andaban cuando empiezan nuestros datos no se tocan. Todas esas filas se marcan con `fila_completa = False`. Los modelos usan solo las filas completas, así todos se comparan con exactamente las mismas filas.

In [0]:
demanda = gold.asignar_particion(gold.agregar_rezagos(base), config.PARTICIONES)
# Marca los días en que la línea todavía no había arrancado (o arrancó hace menos de 28 días)
demanda = gold.linea_en_marcha(demanda, agregado, fecha_inicio)

# Una fila está completa si tiene todos los rezagos, el clima pronosticado y la línea ya andaba.
# Las primeras semanas de 2024 quedan incompletas (no tienen rezago de 28 días ni
# pronóstico antes del 19/01). Los modelos usan solo filas completas.
VARIABLES_REQUERIDAS = [
    "pasajeros_lag_1d", "pasajeros_lag_7d", "pasajeros_lag_14d", "pasajeros_lag_21d",
    "pasajeros_lag_28d", "pasajeros_prom_4sem", "pasajeros_mediana_4sem", "pasajeros_linea_dia_anterior",
    "temperatura_pron", "lluvia_pron_mm", "llueve_pron", "lluvia_pron_dia_mm",
]
demanda = gold.marcar_fila_completa(demanda, VARIABLES_REQUERIDAS,
                                    condicion_extra=F.col("linea_en_marcha"))

COLUMNAS = [
    # identificadores
    "linea", "fecha", "hora", "particion", "fila_completa", "linea_en_marcha",
    # lo que queremos predecir
    "pasajeros",
    # variables de calendario
    "anio", "mes", "dia_mes", "dia_semana", "es_fin_de_semana",
    "es_feriado", "es_no_laborable", "es_carnaval", "es_turismo", "es_eleccion", "es_dia_sin_servicio", "es_paro",
    "es_vispera_navidad", "es_fin_de_anio", "es_semana_fiestas",
    "clases_primaria", "clases_media", "nombre_dia_especial",
    # calendario del día anterior (para saber qué tipo de día fue ayer)
    "ayer_es_no_laborable", "ayer_es_carnaval", "ayer_es_turismo",
    # variables de la línea
    "tipo_linea",
    # clima pronosticado con 24 h de anticipación
    "temperatura_pron", "lluvia_pron_mm", "llueve_pron", "lluvia_pron_dia_mm",
    # rezagos
    "pasajeros_lag_1d", "pasajeros_lag_7d", "pasajeros_lag_14d", "pasajeros_lag_21d", "pasajeros_lag_28d",
    "pasajeros_prom_4sem", "pasajeros_mediana_4sem", "pasajeros_linea_dia_anterior",
    # solo informativa. NO usar en el modelo, porque es del mismo día que se quiere predecir
    "validaciones",
]

(
    demanda.select(*COLUMNAS)
    .write.format("delta").mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(config.TABLE_GOLD)
)
print("Tabla escrita:", config.TABLE_GOLD)

Tabla escrita: bondis_stm.datos.demanda_gold


## 5. Revisiones

Tres controles para estar seguros de que gold está bien:
1. Hay exactamente una fila por cada línea, día y hora (líneas x días x 24).
2. No hay ninguna combinación línea, día y hora repetida.
3. La suma de pasajeros de gold es igual a la de silver para las líneas incluidas. Si no da igual, se perdieron o se duplicaron pasajeros en el camino.

In [0]:
g = spark.table(config.TABLE_GOLD)

n_lineas = g.select("linea").distinct().count()
n_fechas = g.select("fecha").distinct().count()
n_filas = g.count()
print(f"Líneas: {n_lineas}, Fechas: {n_fechas}, Horas: 24")
print(f"Filas:  {n_filas:,}  (esperadas: {n_lineas * n_fechas * 24:,})")
assert n_filas == n_lineas * n_fechas * 24, "La grilla no está completa"

dup = g.groupBy("linea", "fecha", "hora").count().where("count > 1").count()
assert dup == 0, f"Hay {dup} combinaciones línea-fecha-hora repetidas"

pas_gold = g.agg(F.sum("pasajeros")).first()[0]
pas_incl = lineas.where("incluida").agg(F.sum("pasajeros_total")).first()[0]
assert pas_gold == pas_incl, f"Los pasajeros no cuadran: gold {pas_gold:,} vs líneas incluidas {pas_incl:,}"
print(f"Pasajeros en gold: {pas_gold:,} (cuadra con silver para las líneas incluidas)")

Líneas: 136, Fechas: 1035, Horas: 24
Filas:  3,378,240  (esperadas: 3,378,240)
Pasajeros en gold: 843,853,308 (cuadra con silver para las líneas incluidas)


In [0]:
# Por cada conjunto (train, val, test): filas totales, filas completas (las que usa el modelo),
# filas de líneas que todavía no habían arrancado y porcentaje de horas con 0 pasajeros
display(
    g.groupBy("particion")
    .agg(F.min("fecha").alias("desde"), F.max("fecha").alias("hasta"),
         F.min(F.when(F.col("fila_completa"), F.col("fecha"))).alias("primera_fecha_completa"),
         F.count("*").alias("filas"),
         F.sum(F.col("fila_completa").cast("int")).alias("filas_completas"),
         F.sum((~F.col("linea_en_marcha")).cast("int")).alias("filas_linea_sin_arrancar"),
         F.sum("pasajeros").alias("pasajeros"),
         F.round(F.avg((F.col("pasajeros") == 0).cast("int")) * 100, 1).alias("pct_horas_en_cero"))
    .orderBy("desde")
)

particion,desde,hasta,primera_fecha_completa,filas,filas_completas,filas_linea_sin_arrancar,pasajeros,pct_horas_en_cero
train,2024-01-01,2024-12-31,2024-01-29,1194624,1097424,97200,306259563,24.3
val,2025-01-01,2025-12-31,2025-01-01,1191360,1191360,0,298019912,24.7
test,2026-01-01,2026-10-31,2026-01-01,992256,992256,0,239573833,25.5


### Días raros

Los 15 días con menos pasajeros en todo el sistema, solo en train y val (2024 y 2025). Test no se mira, para no decidir variables con él. Sirve para ver si el calendario los explica (feriados, Turismo) o si hay días bajos sin explicación (por ejemplo, un paro).

In [0]:
display(
    g.where(F.col("particion").isin("train", "val"))
    .groupBy("fecha", "dia_semana", "nombre_dia_especial")
    .agg(F.sum("pasajeros").alias("pasajeros"),
         F.countDistinct(F.when(F.col("pasajeros") > 0, F.col("linea"))).alias("lineas_con_servicio"))
    .orderBy("pasajeros")
    .limit(15)
)

fecha,dia_semana,nombre_dia_especial,pasajeros,lineas_con_servicio
2024-05-01,3,Día de los Trabajadores,116,21
2025-05-01,4,Día de los Trabajadores,135,24
2026-05-01,5,Día de los Trabajadores,136,22
2025-01-01,3,Año Nuevo,139,28
2024-01-01,1,Año Nuevo,150,22
2026-01-01,4,Año Nuevo,177,26
2024-12-25,3,Navidad,62855,92
2025-12-25,4,Navidad,69340,92
2026-08-25,2,Declaratoria de la Independencia,195550,115
2026-07-19,7,null,215497,117


In [0]:
# Ejemplo para entender los rezagos: línea 103, martes 12 de marzo de 2024, hora por hora
display(
    g.where((F.col("linea") == "103") & (F.col("fecha") == "2024-03-12"))
    .select("hora", "pasajeros", "pasajeros_lag_1d", "pasajeros_lag_7d", "pasajeros_prom_4sem",
            "pasajeros_mediana_4sem", "pasajeros_linea_dia_anterior")
    .orderBy("hora")
)

hora,pasajeros,pasajeros_lag_1d,pasajeros_lag_7d,pasajeros_prom_4sem,pasajeros_mediana_4sem,pasajeros_linea_dia_anterior
0,193,248,228,195.0,208.0,32083
1,50,152,150,108.25,105.5,32083
2,47,62,75,53.75,52.0,32083
3,33,24,32,29.5,32.0,32083
4,286,272,305,285.0,285.0,32083
5,851,858,839,706.5,689.5,32083
6,1898,1839,1932,1224.5,1047.5,32083
7,2622,2504,2712,1664.5,1550.5,32083
8,2242,2347,2230,1562.0,1613.5,32083
9,1549,1822,1751,1335.5,1457.0,32083


### ¿Las clases cambian la cantidad de pasajeros?

Promedio de pasajeros por día hábil (sin feriados ni Turismo, solo train y val), según si hay clases o no. Si la variable sirve, los días con clases tienen que tener bastantes más pasajeros.

In [0]:
display(
    g.where(~F.col("es_fin_de_semana") & ~F.col("es_feriado") & ~F.col("es_turismo")
            & F.col("particion").isin("train", "val"))
    .groupBy("fecha", "clases_primaria", "clases_media")
    .agg(F.sum("pasajeros").alias("pasajeros_dia"))
    .groupBy("clases_primaria", "clases_media")
    .agg(F.count("*").alias("dias"), F.round(F.avg("pasajeros_dia")).alias("pasajeros_promedio_dia"))
    .orderBy("clases_primaria", "clases_media")
)

clases_primaria,clases_media,dias,pasajeros_promedio_dia
false,false,170,831467.0
true,false,31,1000954.0
true,true,498,1037119.0
